# 02_Item_Feature_Master_Assembler_AI_features

This Python script is designed to consolidate various anime-related feature sets (Xin Wei's structured features, Yichuan's synopsis BERT embeddings, Yichuan's semantic tags, and Yuran's review BERT embeddings) into a single master NumPy matrix. This master matrix can then be used for downstream machine learning tasks, such as building a recommendation system or an anime tower model.

Here's a breakdown of the key steps:

1.  **Loading Mappings:** It starts by loading `anime_mapping.json` to create a mapping from raw anime IDs to sequential indices, which is crucial for populating the master matrix efficiently.

2.  **Loading Data & Calculating Dimensions:**
    *   Loads Xin Wei's structured features from `anime_complete_encoded.csv`.
    *   Determines the dimensions for Yichuan's synopsis BERT embeddings from `synopsis_bert.npy`.
    *   Loads Yichuan's semantic tag features from `synopsis_semantic_tags_features.csv`.
    *   Determines the dimensions for Yuran's review BERT embeddings from `review_bert_embeddings.npy`.
    *   Calculates the `total_dim` of the final master matrix.

3.  **Pre-allocating Master Matrix:** A large NumPy array (`master_matrix`) is pre-allocated with zeros, sized to `(num_anime, total_dim)`, to efficiently store all concatenated features.

4.  **Slotting Data into Master Matrix:**
    *   **Xin Wei's Data:** Iterates through `df_xinwei` and places its features into the initial segment of the `master_matrix` based on the anime ID mapping.
    *   **Yichuan's Synopsis BERT Data:** Loads the BERT embeddings, reads `MAL_ID`s from `anime.csv` (used as a base for ordering), and slots these embeddings into the next segment of the `master_matrix`. It includes a check to ensure length consistency.
    *   **Yichuan's Semantic Tag Features:** Iterates through `df_semantic` and slots these features into their designated segment.
    *   **Yuran's Review Embeddings:** Loads review BERT embeddings and their corresponding anime IDs, then slots them into the final segment of the `master_matrix`.

5.  **Memory Management:** After slotting each feature set, the corresponding pandas DataFrames or NumPy arrays are deleted, and garbage collection (`gc.collect()`) is explicitly called to free up RAM.

6.  **Saving Master Matrix:** Finally, the fully populated `master_matrix` is saved as a NumPy `.npy` file to Google Drive (`MASTER_ANIME_TOWER_FEATURES_WITH_SEMANTICS.npy`).

7.  **Feature Block Boundaries:** The script prints the start and end indices for each feature block within the `master_matrix`, which is useful for later slicing and model input.

In [15]:
import pandas as pd
import numpy as np
import json
import gc

folder_path = '/content/drive/MyDrive'
eng_path = f'{folder_path}/engineered data'

print("1. Loading Mappings...")
with open(f'{folder_path}/anime_mapping.json', 'r') as f:
    anime_mapping = json.load(f)

num_anime = len(anime_mapping)

# ensure integer mapping
id_to_idx = {int(k): int(v) for k, v in anime_mapping.items()}

print("2. Loading Xin Wei's Data & Calculating Dimensions...")

# ============================================================
# XIN WEI FEATURES
# ============================================================
df_xinwei = pd.read_csv(f'{folder_path}/anime_complete_encoded.csv', low_memory=False)
id_col_x = 'MAL_ID' if 'MAL_ID' in df_xinwei.columns else 'anime_id'

df_xinwei = df_xinwei.set_index(id_col_x).select_dtypes(include=[np.number]).astype(np.float32)
xinwei_dim = df_xinwei.shape[1]

# ============================================================
# YICHUAN SYNOPSIS BERT
# ============================================================
yichuan_dim = np.load(f'{eng_path}/synopsis_bert.npy', mmap_mode='r').shape[1]

# ============================================================
# YICHUAN SEMANTIC TAGS (USE FEATURES CSV, NOT RAW CSV)
# ============================================================
semantic_csv_path = f'{eng_path}/synopsis_semantic_tags_features.csv'
df_semantic = pd.read_csv(semantic_csv_path)

if 'MAL_ID' not in df_semantic.columns:
    raise ValueError("synopsis_semantic_tags_features.csv must contain MAL_ID column")

df_semantic = df_semantic.set_index('MAL_ID').select_dtypes(include=[np.number]).astype(np.float32)
semantic_dim = df_semantic.shape[1]

# ============================================================
# YURAN REVIEW EMBEDDING
# ============================================================
yuran_dim = np.load(f'{eng_path}/review_bert_embeddings.npy', mmap_mode='r').shape[1]

# ============================================================
# TOTAL DIMENSION
# ============================================================
total_dim = xinwei_dim + yichuan_dim + semantic_dim + yuran_dim

print(f"   -> Xin Wei structured features: {xinwei_dim}")
print(f"   -> Yichuan synopsis_bert features: {yichuan_dim}")
print(f"   -> Yichuan semantic tag features: {semantic_dim}")
print(f"   -> Yuran review features: {yuran_dim}")
print(f"Total Features per Anime: {total_dim}")

print("\n3. Pre-allocating the Master Matrix in RAM...")
master_matrix = np.zeros((num_anime, total_dim), dtype=np.float32)

# ============================================================
# 4. SLOT XIN WEI
# ============================================================
print("\n4. Slotting Xin Wei's data...")
for raw_id, row in df_xinwei.iterrows():
    raw_id = int(raw_id)
    if raw_id in id_to_idx:
        master_matrix[id_to_idx[raw_id], 0:xinwei_dim] = row.values

del df_xinwei
gc.collect()
print("   -> Xin Wei's data slotted and RAM cleared.")

# ============================================================
# 5. SLOT YICHUAN SYNOPSIS_BERT
# ============================================================
print("\n5. Loading and slotting Yichuan's synopsis_bert data...")
yichuan_embs = np.load(f'{eng_path}/synopsis_bert.npy').astype(np.float32)

# IMPORTANT:
# This assumes synopsis_bert.npy row order matches anime_complete.csv row order
df_base_ids = pd.read_csv(f'{folder_path}/anime.csv', usecols=['MAL_ID'])

if len(df_base_ids) != len(yichuan_embs):
    raise ValueError(
        f"Length mismatch: anime.csv has {len(df_base_ids)} rows, "
        f"but synopsis_bert.npy has {len(yichuan_embs)} rows."
    )

bert_start = xinwei_dim
bert_end = bert_start + yichuan_dim

for i, raw_id in enumerate(df_base_ids['MAL_ID']):
    raw_id = int(raw_id)
    if raw_id in id_to_idx:
        master_matrix[id_to_idx[raw_id], bert_start:bert_end] = yichuan_embs[i]

del yichuan_embs
gc.collect()
print("   -> Yichuan's synopsis_bert data slotted and RAM cleared.")

# ============================================================
# 6. SLOT YICHUAN SEMANTIC TAG FEATURES
# ============================================================
print("\n6. Slotting Yichuan's semantic tag features...")
semantic_start = bert_end
semantic_end = semantic_start + semantic_dim

for raw_id, row in df_semantic.iterrows():
    raw_id = int(raw_id)
    if raw_id in id_to_idx:
        master_matrix[id_to_idx[raw_id], semantic_start:semantic_end] = row.values

del df_semantic
gc.collect()
print("   -> Yichuan's semantic tag features slotted and RAM cleared.")

# ============================================================
# 7. SLOT YURAN REVIEW EMBEDDINGS
# ============================================================
print("\n7. Loading and slotting Yuran's review data...")
review_embs = np.load(f'{eng_path}/review_bert_embeddings.npy').astype(np.float32)
review_ids = np.load(f'{eng_path}/review_anime_ids.npy')

review_start = semantic_end
review_end = review_start + yuran_dim

for i, raw_id in enumerate(review_ids):
    raw_id = int(raw_id)
    if raw_id in id_to_idx:
        master_matrix[id_to_idx[raw_id], review_start:review_end] = review_embs[i]

del review_embs, review_ids, df_base_ids
gc.collect()
print("   -> Yuran's review data slotted and RAM cleared.")

# ============================================================
# 8. SAVE FINAL MASTER MATRIX
# ============================================================
print("\n8. Saving Master Matrix to Google Drive...")
save_path = f'{folder_path}/MASTER_ANIME_TOWER_FEATURES_WITH_SEMANTICS.npy'
np.save(save_path, master_matrix)

print(f"✅✅✅ SUCCESS! Final Master Matrix saved.")
print(f"Path: {save_path}")
print(f"Final shape: {master_matrix.shape}")

# ============================================================
# 9. OPTIONAL: print block boundaries for later model slicing
# ============================================================
print("\n=== FEATURE BLOCK BOUNDARIES ===")
print(f"Xin Wei structured : [0 : {xinwei_dim}]")
print(f"Synopsis BERT      : [{bert_start} : {bert_end}]")
print(f"Semantic tags      : [{semantic_start} : {semantic_end}]")
print(f"Review BERT        : [{review_start} : {review_end}]")
print(f"Total dim          : {total_dim}")

1. Loading Mappings...
2. Loading Xin Wei's Data & Calculating Dimensions...
   -> Xin Wei structured features: 65
   -> Yichuan synopsis_bert features: 384
   -> Yichuan semantic tag features: 61
   -> Yuran review features: 384
Total Features per Anime: 894

3. Pre-allocating the Master Matrix in RAM...

4. Slotting Xin Wei's data...
   -> Xin Wei's data slotted and RAM cleared.

5. Loading and slotting Yichuan's synopsis_bert data...
   -> Yichuan's synopsis_bert data slotted and RAM cleared.

6. Slotting Yichuan's semantic tag features...
   -> Yichuan's semantic tag features slotted and RAM cleared.

7. Loading and slotting Yuran's review data...
   -> Yuran's review data slotted and RAM cleared.

8. Saving Master Matrix to Google Drive...
✅✅✅ SUCCESS! Final Master Matrix saved.
Path: /content/drive/MyDrive/MASTER_ANIME_TOWER_FEATURES_WITH_SEMANTICS.npy
Final shape: (17562, 894)

=== FEATURE BLOCK BOUNDARIES ===
Xin Wei structured : [0 : 65]
Synopsis BERT      : [65 : 449]
Semanti

In [12]:
import os

# List files (excluding directories) in the specified folder
files_in_drive = [f for f in os.listdir('/content/drive/MyDrive') if os.path.isfile(os.path.join('/content/drive/MyDrive', f))]

if files_in_drive:
    print('Files in /content/drive/MyDrive:')
    for f in files_in_drive:
        print(f)
else:
    print('No files found in /content/drive/MyDrive.')

Files in /content/drive/MyDrive:
animelist.csv
anime.csv
caleb_notebook
MASTER_ANIME_TOWER_FEATURES_1BASED.npy
test.csv
val.csv
train.csv
Data_Loader.py
anime_id_map.json
fraud_dataset.csv
raw_df.csv
fraud.ipynb
anime_tower_1_based.npy
caleb_2.ipynb
watch status.ipynb
Untitled0.ipynb
anime_mapping.json
anime data with ai features.ipynb
anime_complete_encoded.csv
synopsis_semantic_tags.npy
synopsis_semantic_tags_features.csv
synopsis_semantic_tags_raw.csv


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
